In [3]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_classic.vectorstores import Chroma

In [5]:
chunk_size = 300
chunk_overlap = 100
loader = WikipediaLoader(query = "Pyramids of Giza", load_max_docs=5)

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
docs = text_splitter.split_documents(documents=documents)
docs



[Document(metadata={'title': 'Great Pyramid of Giza', 'summary': 'The Great Pyramid of Giza is the largest Egyptian pyramid. It served as the tomb of pharaoh Khufu ("Cheops"), who ruled during the Fourth Dynasty of the Old Kingdom. Built c.\u20092600 BC over a period of about 26 years, the pyramid is the oldest of the Seven Wonders of the Ancient World, and the only wonder that has remained largely intact. It is the most famous monument of the Giza pyramid complex, which is part of the UNESCO World Heritage Site "Memphis and its Necropolis". It is situated at the northeastern end of the line of the three main pyramids at Giza.\nInitially standing at 146.6 metres (481 feet), the Great Pyramid was the world\'s tallest human-made structure for more than 3,800 years. Over time, most of the smooth white limestone casing was removed, which lowered the pyramid\'s height to the current 138.5 metres (454.4 ft); what is seen today is the underlying core structure. Each side of the base was measu

In [12]:
vectorstore = Chroma.from_documents(docs, HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2"), persist_directory="output/query_wiki",collection_name="pyramids_of_giza")
retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs = {"k":5, "lambda_mult":0.7})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4991.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_model('groq:openai/gpt-oss-120b', temperature=0.7)

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate

def get_hyde_doc(query):
    template = """Imagine you are an expert writing a detailed explaination on the topic: '{query}'
    create hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template= template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query=query).to_messages()

    print(messages)

    response = llm.invoke(messages)
    hypo_doc = response.content
    return hypo_doc



In [16]:
query = "Who made the pyramid of Giza and how was it built?"
print(get_hyde_doc(query))

[SystemMessage(content="Imagine you are an expert writing a detailed explaination on the topic: 'Who made the pyramid of Giza and how was it built?'\n    create hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
**Who built the Great Pyramid of Giza and how was it erected? – A Detailed, Hypothetical Reconstruction**

*Prepared as if spoken by a senior Egyptologist addressing a mixed audience of scholars, students, and curious lay‑readers.*

---

## 1. The Historical Consensus – Who Was the Builder?

| Item | Evidence | Interpretation |
|------|----------|----------------|
| **Attribution in ancient records** | The *Papyrus Westcar* (c. 1800 BCE) and the *Pyramid Texts* (inscribed inside the chambers) name **Khufu** (also called Cheops) as the owner of the “Mighty Pyramid.” | The ancient Egyptians themselves identified the monument with Khufu, the Fourth Dynasty ruler (c. 2589–2566 BCE). |
| **Quarry marks and mason’s graffiti** | Over 200 limestone blocks 

In [17]:
matched_doc = retriever.invoke(get_hyde_doc(query))
print(matched_doc)

[SystemMessage(content="Imagine you are an expert writing a detailed explaination on the topic: 'Who made the pyramid of Giza and how was it built?'\n    create hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Great_Pyramid_of_Giza', 'title': 'Great Pyramid of Giza', 'summary': 'The Great Pyramid of Giza is the largest Egyptian pyramid. It served as the tomb of pharaoh Khufu ("Cheops"), who ruled during the Fourth Dynasty of the Old Kingdom. Built c.\u20092600 BC over a period of about 26 years, the pyramid is the oldest of the Seven Wonders of the Ancient World, and the only wonder that has remained largely intact. It is the most famous monument of the Giza pyramid complex, which is part of the UNESCO World Heritage Site "Memphis and its Necropolis". It is situated at the northeastern end of the line of the three main pyramids at Giza.\nInitially standing at 146.6 metres (481 feet), the Great 

## HYDE USING LANGCHAIN

In [19]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder
from langchain_classic.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [20]:
loader = TextLoader("Langchain_crewai.txt")

In [21]:
doc = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(doc)

In [22]:
chunks

[Document(metadata={'source': 'Langchain_crewai.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'Langchain_crewai.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'Langchain_crewai.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard'),
 Document(meta

In [23]:
base_embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm = llm,
    base_embeddings = base_embeddings,
    prompt_key = "web_search"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6386.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
vectorstore = Chroma.from_documents(documents=chunks, embedding = hyde_embedding_function, persist_directory="output/hyde_langchain", collection_name="hyde_langchain_crewai")



In [25]:
rag_prompt = PromptTemplate.from_template(
    """
    Use the context below to answer the question.

    Context:
    {context}

    Question: {input}
    """
)

rag_chain = create_stuff_documents_chain(llm = llm, prompt = rag_prompt)

In [26]:
def Hyde_rag_pipeline(query):
    matched_doc = vectorstore.similarity_search(query, k=4)
    response = rag_chain.invoke({"input": query, "context": matched_doc})
    return response

In [27]:
query = "How does Langchain use memory and agents compared to CrewAI"
print(Hyde_rag_pipeline(query))

**LangChain vs CrewAI – How They Handle Memory and Agents**

| Aspect | LangChain | CrewAI |
|--------|-----------|--------|
| **Core idea** | A library that lets you **wire together** language‑model calls, **tools**, and **memory** in a single “agent” pipeline. | A framework that focuses on **building multi‑agent “crews”** where each member has a specialized role (e.g., researcher, writer, reviewer) and the crew orchestrates the overall workflow. |
| **Memory model** | • Provides **explicit, reusable memory objects** (e.g., `ConversationBufferMemory`, `ConversationSummaryMemory`). <br>• Memory is attached to an individual agent and can be swapped or combined (buffer + summary, vector store, etc.). <br>• The memory object decides *what* is stored (raw turns, a summary, embeddings) and *how* it is retrieved on each new LLM call, letting the same LLM stay aware of the full conversation while staying under token limits. | • Memory is **implicit** in the crew’s state. Each crew member rece